# Étape 11 — Pourquoi les graphes LLM prudents aident-ils si peu ?

À l'étape 10, les 978 flèches dont le LLM est le plus sûr ont fait **moins bien** que 978 flèches tirées au hasard dans un même thème,
alors qu'elles contiennent beaucoup plus de vrais prérequis.

**Hypothèse :** le LLM **concentre** ses flèches sur quelques exercices de base, et laisse beaucoup d'exercices sans aucun prérequis.
Or PrereqKT ne peut utiliser le graphe que si l'exercice à prédire **a des voisins**.

Ce notebook mesure, pour chaque graphe utilisé aux étapes 7 à 10 :
- la **couverture** : part des exercices qui ont au moins un prérequis ;
- la **couverture des premières tentatives** : part des premières tentatives du test portant sur un exercice qui a au moins un prérequis (c'est ce que voit le modèle) ;
- la **concentration** : part des flèches qui partent des 10 exercices les plus « prérequis », et indice de Gini des degrés sortants ;

puis il met ces mesures en regard du **bénéfice conservé** pour le KT.

**Aucun entraînement, pas de GPU.** Durée : 2 à 5 minutes.

In [ ]:
# Cellule 1 — Chemins et données
from google.colab import drive          # sur votre propre machine : supprimez ces deux lignes
drive.mount('/content/drive')

import os, random
import numpy as np
import pandas as pd
import networkx as nx
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'   # sur votre propre machine : le chemin de votre dossier projet_kt
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')

exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = [tuple(map(int, e)) for e in zip(*np.nonzero(A_expert))]
scores = pd.read_csv(os.path.join(RESULTATS, 'junyi_llm_scores.csv'))
scores[['A', 'B']] = scores[['A', 'B']].astype(int)
benef = pd.read_csv(os.path.join(RESULTATS, 'rq2_benefice_final.csv'))
print(f'{N} exercices | {len(aretes_expert)} flèches expertes | {len(scores):,} paires évaluées par le LLM')

In [ ]:
# Cellule 2 — Reconstruction exacte des graphes des étapes 7 à 10 (mêmes graines, même ordre)
GRAINES = [0, 1, 2, 3, 4]

# Paires candidates « même thème » (code identique à l'étape 10, cellule C1)
groupes = exos.dropna(subset=['topic']).groupby('topic')['exo_id'].apply(list)
candidats = [(a, b) for ids in groupes for a in ids for b in ids if a != b]

def g_aleatoire(graine):                 # identique à l'étape 7
    rng = random.Random(graine); E = set()
    while len(E) < len(aretes_expert):
        a, b = rng.sample(range(N), 2); E.add((a, b))
    return sorted(E)

def g_theme(n, graine):                  # identique à l'étape 10, cellule C4
    return random.Random(50_000 + graine).sample(candidats, min(n, len(candidats)))

s = scores.sort_values('p_oui', ascending=False).reset_index(drop=True)   # identique à l'étape 9, cellule B4
E_star = len(aretes_expert)
LLM = {f'LLM_x{k}': list(zip(s.A.head(int(k * E_star)), s.B.head(int(k * E_star)))) for k in [0.5, 1, 2, 4, 8]}
LLM['LLM_seuil0.5'] = list(zip(s.A[s.p_oui >= 0.5], s.B[s.p_oui >= 0.5]))

GRAPHES = {'expert': [aretes_expert]}
GRAPHES['aleatoire'] = [g_aleatoire(g) for g in GRAINES]
for lib, n in {'theme_978': 978, 'theme_3912': 3912, 'theme_7176': len(LLM['LLM_seuil0.5']), 'theme_7824': 7824}.items():
    GRAPHES[lib] = [g_theme(n, g) for g in GRAINES]
for lib, E in LLM.items():
    GRAPHES[lib] = [E]
print({k: len(v[0]) for k, v in GRAPHES.items()})

In [ ]:
# Cellule 3 — Exercices des premières tentatives du test (ce que le modèle doit prédire)
test = pd.read_csv(os.path.join(PREP, 'sequences_test.csv')).sort_values(['uid', 'morceau'])
premieres = []
vus = {}
for uid, ex in zip(test['uid'], test['exercices']):
    deja = vus.setdefault(uid, set())
    for x in map(int, ex.split(',')):
        if x not in deja:
            deja.add(x); premieres.append(x)
premieres = pd.Series(premieres).value_counts()
print(f'{premieres.sum():,} premières tentatives dans le test, sur {len(premieres)} exercices')

In [ ]:
# Cellule 4 — Couverture et concentration de chaque graphe
def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    if x.sum() == 0:
        return 0.0
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))

def mesures(E):
    E = list(set(map(tuple, E)))
    deg_in, deg_out = np.zeros(N), np.zeros(N)
    for a, b in E:
        deg_out[a] += 1; deg_in[b] += 1
    a_prereq = deg_in > 0
    a_voisin = (deg_in + deg_out) > 0
    top10 = np.sort(deg_out)[::-1][:10].sum()
    poids = premieres.reindex(range(N), fill_value=0).values
    return {
        'fleches': len(E),
        'couverture_prerequis_%': 100 * a_prereq.mean(),
        'couverture_voisins_%': 100 * a_voisin.mean(),
        'couverture_1res_tentatives_%': 100 * (poids * a_prereq).sum() / poids.sum(),
        'part_fleches_top10_sources_%': 100 * top10 / max(1, len(E)),
        'gini_degre_sortant': gini(deg_out),
        'prerequis_par_exercice_couvert': deg_in[a_prereq].mean() if a_prereq.any() else 0.0,
    }

lignes = []
for lib, versions in GRAPHES.items():
    m = pd.DataFrame([mesures(E) for E in versions]).mean()
    lignes.append({'graphe': lib, **m.round(2).to_dict()})
couv = pd.DataFrame(lignes)
couv['fleches'] = couv['fleches'].astype(int)
couv.to_csv(os.path.join(RESULTATS, 'rq2_couverture_graphes.csv'), index=False)
print(couv.to_string(index=False))

# Les exercices vers lesquels le LLM prudent fait partir le plus de flèches
E = LLM['LLM_x1']
top = pd.Series([a for a, _ in E]).value_counts().head(10)
print('\nLLM_x1 : les 10 exercices « source » les plus fréquents')
print(pd.DataFrame({'exercice': exos.set_index('exo_id').loc[top.index, 'name'].values,
                    'fleches_sortantes': top.values}).to_string(index=False))

In [ ]:
# Cellule 5 — Lien entre couverture et bénéfice pour le KT (corrélation de Spearman sur les graphes)
import matplotlib.pyplot as plt
mesure = 'couverture_1res_tentatives_%'
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, frac in zip(axes, [0.05, 0.01]):
    d = benef[benef.fraction == frac].merge(couv, on='graphe')
    d = d[d.graphe != 'aucun']
    rho, p = stats.spearmanr(d[mesure], d['benefice_conserve_%'])
    rho2, p2 = stats.spearmanr(d['part_fleches_top10_sources_%'], d['benefice_conserve_%'])
    print(f'{frac:.0%} des élèves : Spearman(couverture 1res tentatives, bénéfice) = {rho:.2f} (p = {p:.3f}) | '
          f'Spearman(concentration top 10, bénéfice) = {rho2:.2f} (p = {p2:.3f}) | {len(d)} graphes')
    couleurs = d.graphe.map(lambda g: 'tab:blue' if g.startswith('LLM') else 'tab:orange' if g.startswith('theme')
                            else 'black' if g == 'expert' else 'grey')
    ax.scatter(d[mesure], d['benefice_conserve_%'], c=couleurs, s=50, zorder=3)
    for _, r in d.iterrows():
        ax.annotate(r.graphe.replace('LLM_', '').replace('theme_', 'th'), (r[mesure], r['benefice_conserve_%']),
                    fontsize=7, xytext=(3, 3), textcoords='offset points')
    ax.axhline(0, ls=':', c='black', lw=1)
    ax.set_xlabel('Premières tentatives portant sur un exercice avec prérequis (%)')
    ax.set_title(f'{frac:.0%} des élèves — Spearman ρ = {rho:.2f}')
axes[0].set_ylabel('Bénéfice conservé (%)')
fig.tight_layout()
fig.savefig(os.path.join(RESULTATS, 'rq2_couverture_vs_benefice.png'), dpi=200)
plt.show()